# ACSIncome: geographic and temporal exploration

All 50 US states, 2018 and 2021, one-year person survey. DC and Puerto Rico are excluded. Descriptive exploration only: no model fitting or split selection.

Run with the project `.venv` kernel. Downloads are cached in `exploration/data`. Tables are saved in `exploration/eda_outputs`. Counts and rates describe the filtered survey sample; weighted rates use person weights. The standard label compares unadjusted PINCP to $50,000, so temporal differences are not constant-dollar comparisons.

## Results from the completed run

Coverage: **50 states × 2 years = 100 combinations**. All code cells executed successfully.

| Scope | Retained rows, 2018 | Retained rows, 2021 | Positive rate, 2018 | Positive rate, 2021 |
|---|---:|---:|---:|---:|
| 50-state pooled survey sample | 1,655,429 | 1,621,285 | 37.04% | 41.99% |
| California | 195,665 | 192,223 | 41.06% | 45.77% |

These are unweighted sample rates. California person-weighted rates are 38.43% and 44.11%, respectively. The sample positive-rate range across states is 26.80%–49.04% in 2018 and 29.64%–53.21% in 2021.

No filtered feature values were missing before Folktables conversion in any of the 100 combinations. Observed category-code sets match across the pooled years except POBP code 464, observed only in 2018; this absence alone does not establish a coding-definition change. Relationship codes match only after the explicitly documented harmonization below.

The pooled positive-rate increase is **4.95 percentage points**. It does not by itself establish conditional shift, and the fixed dollar threshold is not adjusted for inflation. All tables are saved in `eda_outputs/`; category summaries include counts, shares, and within-category label rates.


In [1]:
from pathlib import Path
import json
import platform
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display
from folktables import ACSDataSource, ACSIncome
from folktables.acs import adult_filter

BASE = Path.cwd() if Path.cwd().name == "exploration" else Path.cwd() / "exploration"
CACHE = BASE / "data"
OUT = BASE / "eda_outputs"
OUT.mkdir(parents=True, exist_ok=True)
STATES = "AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY".split()
YEARS = ["2018", "2021"]
FEATURES = list(ACSIncome.features)
NUMERICAL = ["AGEP", "WKHP"]
CATEGORICAL = [f for f in FEATURES if f not in NUMERICAL]
metadata = {"python": platform.python_version(), "folktables": importlib.metadata.version("folktables"), "states": STATES, "years": YEARS, "horizon": "1-Year", "survey": "person", "label": "PINCP > 50000 (no inflation adjustment)"}
(OUT / "run_metadata.json").write_text(json.dumps(metadata, indent=2))
print(metadata)


{'python': '3.12.13', 'folktables': '0.0.12', 'states': ['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ', 'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY'], 'years': ['2018', '2021'], 'horizon': '1-Year', 'survey': 'person', 'label': 'PINCP > 50000 (no inflation adjustment)'}


## Feature meanings and selection

Numerical: age and usual weekly work hours. SCHL is ordinal categorical; the other coded features are nominal. Numeric dtype does not make category codes quantities.

Keep AGEP > 16, PINCP > 100, WKHP > 0, PWGTP >= 1. The boolean target is PINCP > 50000. RAC1P is both a feature and the returned group. Folktables replaces feature NaNs with zero during conversion, so inspect the dataframe first.

Sources: [Folktables ACSIncome](https://github.com/socialfoundations/folktables/blob/main/folktables/acs.py), [Census PUMS documentation](https://www.census.gov/programs-surveys/acs/microdata/documentation.html).

In [2]:
meanings = {"AGEP": "Age", "COW": "Class of worker", "SCHL": "Educational attainment", "MAR": "Marital status", "OCCP": "Occupation", "POBP": "Place of birth", "RELP": "Relationship to reference person", "WKHP": "Usual weekly work hours", "SEX": "Sex", "RAC1P": "Race recode"}
feature_table = pd.DataFrame([{"feature": f, "meaning": meanings[f], "type": "numerical" if f in NUMERICAL else "ordinal categorical" if f == "SCHL" else "nominal categorical"} for f in FEATURES])
display(feature_table)
feature_table.to_csv(OUT / "feature_dictionary.csv", index=False)

,feature,meaning,type
0,AGEP,Age,numerical
1,COW,Class of worker,nominal categorical
2,SCHL,Educational attainment,ordinal categorical
3,MAR,Marital status,nominal categorical
4,OCCP,Occupation,nominal categorical
5,POBP,Place of birth,nominal categorical
6,RELP,Relationship to reference person,nominal categorical
7,WKHP,Usual weekly work hours,numerical
8,SEX,Sex,nominal categorical
9,RAC1P,Race recode,nominal categorical


## Download cache

Up to four downloads run concurrently, with bounded retries, HTTP status checks and ZIP integrity validation. Existing CSVs are reused.

In [3]:
# Download safely with retries; extract only the expected CSV after ZIP validation.
import importlib
import time
import zipfile
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
state_codes = importlib.import_module("folktables.load_acs")._STATE_CODES

def ensure_cached(year, state):
    folder = CACHE / year / "1-Year"
    folder.mkdir(parents=True, exist_ok=True)
    filename = f"psam_p{state_codes[state]}.csv"
    target = folder / filename
    if target.exists() and target.stat().st_size > 0:
        return f"Cached {year} {state}"
    url = f"https://www2.census.gov/programs-surveys/acs/data/pums/{year}/1-Year/csv_p{state.lower()}.zip"
    temporary = folder / f"{state}.download.zip"
    for attempt in range(5):
        try:
            with requests.get(url, stream=True, timeout=(30, 180)) as response:
                response.raise_for_status()
                with temporary.open("wb") as handle:
                    for chunk in response.iter_content(1024*1024):
                        handle.write(chunk)
            with zipfile.ZipFile(temporary) as archive:
                assert archive.testzip() is None, "Corrupted ZIP"
                with archive.open(filename) as source_file, target.with_suffix(".partial").open("wb") as dest:
                    import shutil
                    shutil.copyfileobj(source_file, dest)
            target.with_suffix(".partial").replace(target)
            temporary.unlink(missing_ok=True)
            return f"Downloaded {year} {state}"
        except Exception:
            temporary.unlink(missing_ok=True)
            target.with_suffix(".partial").unlink(missing_ok=True)
            if attempt == 4:
                raise
            time.sleep(min(2**attempt, 16))

with ThreadPoolExecutor(max_workers=4) as pool:
    jobs = [pool.submit(ensure_cached, year, state) for year in YEARS for state in STATES]
    with (OUT / "download_progress.txt").open("w") as progress:
        for job in as_completed(jobs):
            message = job.result()
            print(message, flush=True)
            progress.write(message + "\n")
            progress.flush()
print("All 100 source CSVs are cached.")

Cached 2018 AK


Cached 2018 AZ


Cached 2018 AL


Cached 2018 AR


Cached 2018 CA


...

Cached 2021 VA


Cached 2021 WY


Cached 2021 UT


Cached 2021 WA


All 100 source CSVs are cached.


## Load and summarize each state–year

Filter counts are sequential, so each removal count depends on the preceding filters. Missingness is measured before conversion. Numerical summaries and category shares are unweighted; the weighted positive rate estimates prevalence in the selected population. Rare categories have noisier estimates. Download failures stop execution rather than silently omitting states.

### Relationship harmonization required for 2021

2021 replaces RELP with RELSHIPP. We map detailed spouse/partner categories to the older broad spouse/partner categories, and pool 2018 roomer/boarder (11) into other nonrelative (15) because 2021 has no separate roomer/boarder category. The comparison RELP feature is therefore an explicit harmonized adaptation, not the untouched 2018 benchmark. Original filtered relationship codes are saved separately in `original_relationship_codes.csv`. Missing values remain missing before conversion. Income selection and labels are unchanged.

The mapping is grounded in the official 2018 and 2021 dictionaries saved with the outputs. Other feature definitions still need semantic review; identical numeric codes alone do not establish equivalence.

In [4]:
summary_rows, filter_rows, missing_rows, numeric_rows, category_rows, relationship_rows = [], [], [], [], [], []
relationship_map_2021 = {20:0, 21:1, 22:13, 23:1, 24:13, 25:2, 26:3, 27:4, 28:5, 29:6, 30:7, 31:8, 32:9, 33:10, 34:12, 35:14, 36:15, 37:16, 38:17}
for year in YEARS:
    source = ACSDataSource(survey_year=year, horizon="1-Year", survey="person", root_dir=str(CACHE))
    for state in STATES:
        raw = source.get_data(states=[state], download=False)
        relationship_column = "RELP" if "RELP" in raw else "RELSHIPP"
        original_filtered = adult_filter(raw)
        for value, count in original_filtered[relationship_column].value_counts(dropna=False).items():
            relationship_rows.append(dict(state=state, year=int(year), source_feature=relationship_column, code=value, count=int(count), share=float(count/len(original_filtered))))
        if relationship_column == "RELSHIPP":
            observed_codes = set(raw[relationship_column].dropna().unique())
            assert observed_codes.issubset(relationship_map_2021), observed_codes
            raw["RELP"] = raw[relationship_column].map(relationship_map_2021)
        else:
            # No separate roomer/boarder category in 2021: pool into other nonrelative.
            raw["RELP"] = raw["RELP"].replace({11:15})
        filtered = adult_filter(raw)
        X, y, group = ACSIncome.df_to_numpy(raw)
        assert len(filtered) == len(y) and X.shape[1] == 10
        weights = filtered["PWGTP"].to_numpy(dtype=float)
        summary_rows.append(dict(state=state, year=int(year), raw_rows=len(raw), retained_rows=len(y), retention_rate=len(y)/len(raw), positive_rows=int(y.sum()), positive_rate=float(y.mean()), weighted_positive_rate=float(np.average(y, weights=weights)), person_weight_sum=float(weights.sum()), feature_memory_mb=X.nbytes/1e6, converted_nan_count=int(np.isnan(X).sum())))
        current = raw
        for column, cutoff in [("AGEP", 16), ("PINCP", 100), ("WKHP", 0), ("PWGTP", 1)]:
            before = len(current)
            mask = current[column].ge(cutoff) if column == "PWGTP" else current[column].gt(cutoff)
            current = current.loc[mask]
            filter_rows.append(dict(state=state, year=int(year), rule=f"{column} {'>=' if column == 'PWGTP' else '>'} {cutoff}", before=before, after=len(current), removed=before-len(current)))
        for stage, frame in [("raw", raw), ("filtered", filtered)]:
            for feature in FEATURES + ["PINCP", "PWGTP"]:
                missing_rows.append(dict(state=state, year=int(year), stage=stage, feature=feature, missing_count=int(frame[feature].isna().sum()), missing_rate=float(frame[feature].isna().mean()), zero_count=int(frame[feature].eq(0).sum())))
        for feature in NUMERICAL:
            stats = filtered[feature].describe(percentiles=[.05,.25,.5,.75,.95])
            numeric_rows.append(dict(state=state, year=int(year), feature=feature, **stats.to_dict()))
        labels = pd.Series(y, index=filtered.index)
        for feature in CATEGORICAL:
            table = pd.DataFrame({"code": filtered[feature], "label": labels}).groupby("code", dropna=False)["label"].agg(["size", "mean"])
            for category, row in table.iterrows():
                category_rows.append(dict(state=state, year=int(year), feature=feature, code=category, count=int(row["size"]), share=float(row["size"] / len(y)), positive_rate=float(row["mean"])))
        print(f"{year} {state}: {len(y):,} retained, positive rate {y.mean():.4f}", flush=True)
        del raw, filtered, original_filtered, current, X, y, group, labels, table
summary = pd.DataFrame(summary_rows)
filters = pd.DataFrame(filter_rows)
missingness = pd.DataFrame(missing_rows)
numerical = pd.DataFrame(numeric_rows)
categories = pd.DataFrame(category_rows)
for name, table in [("state_year_summary", summary), ("sequential_filters", filters), ("missingness", missingness), ("numerical_summary", numerical), ("category_summary", categories)]:
    table.to_csv(OUT / f"{name}.csv", index=False)
pd.DataFrame(relationship_rows).to_csv(OUT / "original_relationship_codes.csv", index=False)
assert len(summary) == 100
print("Completed all 100 state–year combinations.")

2018 AL: 22,268 retained, positive rate 0.3109


2018 AK: 3,546 retained, positive rate 0.3866


2018 AZ: 33,277 retained, positive rate 0.3396


2018 AR: 13,929 retained, positive rate 0.2680


2018 CA: 195,665 retained, positive rate 0.4106


...

2021 WA: 39,954 retained, positive rate 0.4941


2021 WV: 7,488 retained, positive rate 0.3427


2021 WI: 31,684 retained, positive rate 0.3991


2021 WY: 2,944 retained, positive rate 0.3723


Completed all 100 state–year combinations.


In [5]:
display(summary.round(4))
national = summary.groupby("year").agg(raw_rows=("raw_rows", "sum"), retained_rows=("retained_rows", "sum"), positive_rows=("positive_rows", "sum"))
national["pooled_sample_positive_rate"] = national.positive_rows / national.retained_rows
display(national.round(4))
national.to_csv(OUT / "pooled_sample_summary.csv")
display(missingness.groupby(["year", "stage", "feature"])["missing_count"].sum().unstack(["year", "stage"]))
display(numerical.head(10).round(3))

,state,year,raw_rows,retained_rows,retention_rate,positive_rows,positive_rate,weighted_positive_rate,person_weight_sum,feature_memory_mb,converted_nan_count
0,AL,2018,47777,22268,0.4661,6924,0.3109,0.2933,2310884.0,1.7814,0
1,AK,2018,6711,3546,0.5284,1371,0.3866,0.4151,406006.0,0.2837,0
2,AZ,2018,69990,33277,0.4755,11300,0.3396,0.3134,3545781.0,2.6622,0
3,AR,2018,30503,13929,0.4566,3733,0.2680,0.2574,1438510.0,1.1143,0
4,CA,2018,378817,195665,0.5165,80335,0.4106,0.3843,20625542.0,15.6532,0
...,...,...,...,...,...,...,...,...,...,...,...
95,VA,2021,87093,45928,0.5273,21952,0.4780,0.4550,4682378.0,3.6742,0
96,WA,2021,78528,39954,0.5088,19743,0.4941,0.4829,4125171.0,3.1963,0
97,WV,2021,17428,7488,0.4297,2566,0.3427,0.3315,805701.0,0.5990,0
98,WI,2021,60281,31684,0.5256,12645,0.3991,0.3956,3242347.0,2.5347,0


,raw_rows,retained_rows,positive_rows,pooled_sample_positive_rate
year,,,,
2018,3207990,1655429,613105,0.3704
2021,3245437,1621285,680722,0.4199


year        2018              2021         
stage   filtered      raw filtered      raw
feature                                    
AGEP           0        0        0        0
COW            0  1305289        0  1324349
MAR            0        0        0        0
OCCP           0  1305289        0  1324349
PINCP          0   531677        0   510002
POBP           0        0        0        0
PWGTP          0        0        0        0
RAC1P          0        0        0        0
RELP           0        0        0        0
SCHL           0    94955        0    87888
SEX            0        0        0        0
WKHP           0  1538967        0  1609633

,state,year,feature,count,mean,std,min,5%,25%,50%,75%,95%,max
0,AL,2018,AGEP,22268.0,43.462,15.422,17.0,20.0,30.0,44.0,56.0,68.0,93.0
1,AL,2018,WKHP,22268.0,38.573,12.617,1.0,15.0,35.0,40.0,42.0,60.0,99.0
2,AK,2018,AGEP,3546.0,41.619,14.954,17.0,20.0,29.0,41.0,54.0,65.0,91.0
3,AK,2018,WKHP,3546.0,42.151,18.737,1.0,12.0,35.0,40.0,48.0,84.0,99.0
4,AZ,2018,AGEP,33277.0,43.113,15.336,17.0,20.0,30.0,43.0,56.0,68.0,93.0
5,AZ,2018,WKHP,33277.0,38.304,12.723,1.0,15.0,35.0,40.0,40.0,60.0,99.0
6,AR,2018,AGEP,13929.0,43.177,15.390,17.0,20.0,30.0,43.0,56.0,68.0,94.0
7,AR,2018,WKHP,13929.0,38.784,12.802,1.0,15.0,36.0,40.0,42.0,60.0,99.0
8,CA,2018,AGEP,195665.0,42.735,14.885,17.0,20.0,30.0,42.0,55.0,67.0,94.0
9,CA,2018,WKHP,195665.0,37.867,13.019,1.0,12.0,32.0,40.0,40.0,60.0,99.0


## Temporal changes and geographic variation

Rate differences below are percentage points, not percentage changes. Total variation (TV) between category-share vectors is half the sum of absolute differences: 0 means identical observed shares, 1 means disjoint support. These are descriptive measures without significance claims. Within-feature category label rates do not condition on all ten features.

In [6]:
wide = summary.pivot(index="state", columns="year", values=["retained_rows", "positive_rate", "weighted_positive_rate"])
temporal = pd.DataFrame(index=wide.index)
for year in [2018, 2021]:
    temporal[f"rows_{year}"] = wide["retained_rows"][year].astype(int)
    temporal[f"positive_rate_{year}"] = wide["positive_rate"][year]
temporal["positive_rate_change_pp"] = 100*(wide["positive_rate"][2021]-wide["positive_rate"][2018])
temporal["weighted_rate_change_pp"] = 100*(wide["weighted_positive_rate"][2021]-wide["weighted_positive_rate"][2018])
display(temporal.sort_values("positive_rate_change_pp", ascending=False).round(4))
temporal.to_csv(OUT / "temporal_comparison.csv")
geographic = summary.groupby("year").agg(min_positive_rate=("positive_rate", "min"), max_positive_rate=("positive_rate", "max"), median_state_positive_rate=("positive_rate", "median"), min_rows=("retained_rows", "min"), max_rows=("retained_rows", "max"))
display(geographic.round(4))
geographic.to_csv(OUT / "geographic_summary.csv")
shift_rows = []
for (state, feature), part in categories.groupby(["state", "feature"]):
    shares = part.pivot(index="code", columns="year", values="share").fillna(0)
    shift_rows.append(dict(state=state, feature=feature, total_variation=float((shares[2021]-shares[2018]).abs().sum()/2)))
category_shift = pd.DataFrame(shift_rows)
display(category_shift.pivot(index="state", columns="feature", values="total_variation").round(4))
category_shift.to_csv(OUT / "temporal_category_shift.csv", index=False)
display(categories.head(20).round(4))

,rows_2018,positive_rate_2018,rows_2021,positive_rate_2021,positive_rate_change_pp,weighted_rate_change_pp
state,,,,,,
NH,7966,0.3999,7516,0.4795,7.9561,7.8029
MN,31021,0.3723,30453,0.4449,7.2554,6.4050
NM,8711,0.2977,8591,0.3647,6.7014,7.1053
OR,21919,0.3645,21370,0.4313,6.6780,6.5657
SD,4899,0.2701,4723,0.3367,6.6595,9.5378
WA,39944,0.4283,39954,0.4941,6.5794,7.2054
NE,10785,0.3123,10676,0.3780,6.5665,6.5373
MT,5463,0.2925,5408,0.3572,6.4735,6.7401
PA,68308,0.3570,65241,0.4203,6.3348,6.7028


,min_positive_rate,max_positive_rate,median_state_positive_rate,min_rows,max_rows
year,,,,,
2018,0.2680,0.4904,0.3396,3064,195665
2021,0.2964,0.5321,0.3886,2944,192223


feature,COW,MAR,OCCP,POBP,RAC1P,RELP,SCHL,SEX
state,,,,,,,,
AK,0.0530,0.0169,0.2140,0.0746,0.0636,0.0420,0.0523,0.0041
AL,0.0221,0.0177,0.1020,0.0275,0.0576,0.0173,0.0289,0.0046
AR,0.0254,0.0086,0.1222,0.0280,0.0971,0.0169,0.0227,0.0011
AZ,0.0174,0.0184,0.1040,0.0375,0.1725,0.0157,0.0343,0.0035
CA,0.0217,0.0161,0.0766,0.0214,0.2037,0.0181,0.0346,0.0030
CO,0.0189,0.0203,0.1039,0.0377,0.1252,0.0181,0.0307,0.0016
CT,0.0241,0.0210,0.1090,0.0476,0.0927,0.0182,0.0329,0.0019
DE,0.0341,0.0190,0.1978,0.0600,0.0898,0.0177,0.0441,0.0081
FL,0.0146,0.0157,0.0810,0.0289,0.2011,0.0128,0.0271,0.0036


,state,year,feature,code,count,share,positive_rate
0,AL,2018,COW,1.0,15013,0.6742,0.2868
1,AL,2018,COW,2.0,1529,0.0687,0.3165
2,AL,2018,COW,3.0,1501,0.0674,0.2678
3,AL,2018,COW,4.0,1289,0.0579,0.3747
4,AL,2018,COW,5.0,796,0.0357,0.6771
5,AL,2018,COW,6.0,1315,0.0591,0.2373
6,AL,2018,COW,7.0,766,0.0344,0.5039
7,AL,2018,COW,8.0,59,0.0026,0.2203
8,AL,2018,SCHL,1.0,166,0.0075,0.1265
9,AL,2018,SCHL,2.0,6,0.0003,0.5000


## Observed code consistency and official definitions

A code observed only in one year is not automatically a changed definition: it may reflect rarity or composition. Conversely, the same code can change meaning. The support table checks observed codes only; consult the downloaded official dictionaries to check meanings and harmonize before pooling. Missing values are excluded from observed code sets and reported separately.

In [7]:
support_rows = []
for feature in CATEGORICAL:
    observed = {year: set(categories.loc[(categories.feature == feature) & (categories.year == year), "code"].dropna()) for year in [2018, 2021]}
    support_rows.append(dict(feature=feature, codes_2018=len(observed[2018]), codes_2021=len(observed[2021]), only_2018=sorted(observed[2018]-observed[2021]), only_2021=sorted(observed[2021]-observed[2018])))
support = pd.DataFrame(support_rows)
display(support)
support.to_csv(OUT / "observed_code_support.csv", index=False)
for year in YEARS:
    definitions = ACSDataSource(survey_year=year, horizon="1-Year", survey="person", root_dir=str(CACHE)).get_definitions(download=True)
    definitions.to_csv(OUT / f"official_definitions_{year}.csv", index=False, header=False)
print("Official dictionaries saved for manual semantic comparison; observed support does not establish code equivalence.")

,feature,codes_2018,codes_2021,only_2018,only_2021
0,COW,8,8,[],[]
1,SCHL,24,24,[],[]
2,MAR,5,5,[],[]
3,OCCP,529,529,[],[]
4,POBP,224,223,[464.0],[]
5,RELP,17,17,[],[]
6,SEX,2,2,[],[]
7,RAC1P,9,9,[],[]


Official dictionaries saved for manual semantic comparison; observed support does not establish code equivalence.


## Interpreting shift

Changes in feature distributions describe covariate differences. Changes in positive rates describe label-prevalence differences. Neither alone establishes a change in P(Y | X).

For example, a population with more people in high-paying occupations can have a higher overall positive rate even if income probabilities within identical feature profiles remain unchanged. Category-specific rates are useful clues, but other features can still differ within a category. Survey composition, coding changes, and the fixed dollar threshold also affect interpretation.

Review the saved tables independently. No states have been assigned training, validation, or hidden roles.